In [1]:
import re
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model

In [2]:
PII_PATTERNS = {
    "email": r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+",
    "phone": r"\b\d{10}\b",
    "credit_card": r"\b(?:\d[ -]*?){13,16}\b",
    "aadhaar": r"\b\d{4}\s?\d{4}\s?\d{4}\b"
}

In [3]:
def pii_guardrail(user_input: str):
    detected = []
    
    for pii_type, pattern in PII_PATTERNS.items():
        matches = re.findall(pattern, user_input)
        if matches:
            detected.append({
                "type": pii_type,
                "matches": matches
            })

    return detected

In [4]:
def safe_chat(user_input):
    pii_found = pii_guardrail(user_input)

    if pii_found:
        return {
            "blocked": True,
            "message": "PII detected in input.",
            "details": pii_found
        }

    llm = init_chat_model("google_genai:gemini-2.5-flash-lite")

    response = llm.invoke([
        HumanMessage(content=user_input)
    ])

    return {
        "blocked": False,
        "response": response.content
    }

In [5]:
query = """
My email is tanish@gmail.com
and my phone number is 9876543210
"""

result = safe_chat(query)

print(result)

{'blocked': True, 'message': 'PII detected in input.', 'details': [{'type': 'email', 'matches': ['tanish@gmail.com']}, {'type': 'phone', 'matches': ['9876543210']}]}
